# Open Information Extraction

L'extraction d'information (EI) consiste à identifier et organiser les informations contenues dans des textes non structurés. Les tâches d'EI sont très variées car elles peuvent cibler différents types d'éléments (entités, relations, événements, sentiments, etc.), et ces éléments peuvent avoir des structures très différentes (*spans*, triplets, etc.). L'extraction d'information ouverte (OIE) est sous-domaine de recherche dont les méthodes visent à extraire des triplets sujet-prédicat-objet à partir de phrases données.

## SVO and SPO extraction

Un triplet sujet-verbe-objet (SVO) est une structure de base en traitement automatique du langage naturel (TAL) qui identifie les composants principaux d'une phrase : le sujet (qui effectue l'action), le verbe (l'action elle-même) et l'objet (qui reçoit l'action).

Ainsi, dans la phrase "Le chat chasse la souris", le triplet SVO serait : (Sujet : "Le chat", Verbe : "chasse", Objet : "la souris"). 

Un triplet sujet-prédicat-objet (SPO) offre une représentation plus détaillée et sémantique des relations entre entités. Le prédicat peut inclure non seulement des verbes, mais aussi des attributs ou des propriétés. 

Par exemple, à partir de la même phrase, on pourrait extraire des triplets SPO supplémentaires : (Sujet : "Le chat", Prédicat : "est", Objet : "un félin") et (Sujet : "La souris", Prédicat : "est", Objet : "un rongeur").

Ces informations sont extraites d'ontologies ou de base de données pré-existantes (en anglais DBpedia, Freebase, Wikidata) par \textit{entity linking}, et l'extraction de relations (entre les différentes entités, comme dans la figure au dessus). Les triplets SPO sont couramment utilisés aujourd'hui dans la construction de graphes de connaissances (*knowledge graph*).

![Source: Wikipedia. Exemple d'un réseau sémantique.](./arbre_semantique.png)

## Implémentation avec Spacy

L'implémentation se fera avec Spacy. Vous pouvez aussi vous aider des papiers suivants~:
- Ash, E., Gauthier, G., & Widmer, P. (2024). Relatio: Text Semantics Capture Political and Economic Narratives. Political Analysis, 32(1), 115–132. Available at : https://www.cambridge.org/core/journals/political-analysis/article/relatio-text-semantics-capture-political-and-economic-narratives/E72C0482A44C9A817E381B394A73E2D6
- Fabrice Lamarche et Philippe Langlais. (2024). BenchIEFL : A Manually Re-Annotated Fact-Based Open Information Extraction Benchmark. In Findings of the Association for Computational Linguistics.  Association for Computational Linguistics. Available at : https://aclanthology.org/2024.findings-acl.496/


In [1]:
import polars as pl
import spacy
from spacy import displacy
from spacy.lang.en import English

if not spacy.tokens.Token.has_extension("noun_chunk"):
    spacy.tokens.Token.set_extension("noun_chunk", default=None)

In [6]:
spacy.explain("B")

/Users/shen/.pyenv/versions/3.11.10/envs/torch/lib/python3.11/site-packages/spacy/glossary.py:20: UserWarning: [W118] Term 'B' not found in glossary. It may however be explained in documentation for the corpora used to train the language. Please check `nlp.meta["sources"]` for any relevant links.
  warnings.warn(Warnings.W118.format(term=term))


# Preprocessing

## split in sentences

On importe le corpus et on rajoute un id à chaque document (article ou publication) s'ils n'en ont pas encore un.

In [2]:
corpus = pl.read_csv("trump_archive.csv")
corpus.head()

id,publication
f64,str
9.8455e16,"""Republicans and Democrats have…"
1.2347e18,"""I was thrilled to be back in t…"
1.3049e18,"""The Unsolicited Mail In Ballot…"
1.2236e18,"""Getting a little exercise this…"
1.2152e18,"""Thank you Elise!"""


In [3]:
corpus = corpus.with_columns(
    pl.int_range(pl.len(), dtype=pl.UInt32).alias("publication_id")
).select(
    ["publication_id", "publication"]
)
corpus.head()

publication_id,publication
u32,str
0,"""Republicans and Democrats have…"
1,"""I was thrilled to be back in t…"
2,"""The Unsolicited Mail In Ballot…"
3,"""Getting a little exercise this…"
4,"""Thank you Elise!"""


On découpe les documents en phrase avec `Spacy`. On donne à chaque phrase l'id du document de provenance.

In [4]:
sents  = []
ids = [] # index du document dans le dataframe

nlp = English()
nlp.add_pipe("sentencizer")

spacy_docs = nlp.pipe(corpus["publication"])

for i, doc in enumerate(spacy_docs):
    for sent in doc.sents:
        sents.append(str(sent))
        ids.append(corpus["publication_id"][i])

In [5]:
len(sents)

63723

In [6]:
ids[:10]

[0, 1, 1, 2, 2, 2, 3, 4, 5, 6]

On crée un nouveau fichier pour notre corpus pour ne pas répéter l'opération à chaque fois.

In [7]:
pl.DataFrame(
    {
        "id": ids,
        "sent": sents
    }
).write_csv("trump_archive_sents.csv")

## Quelques petites manipulations avec Spacy

Pour obtenir un groupe nominal à partir d'un des tokens, vous pouvez utiliser les *noun chunks* de Spacy.

La documentation sur les *noun chunks* : https://spacy.io/usage/linguistic-features/#noun-chunks

In [8]:
nlp = spacy.load("en_core_web_lg")

In [9]:
doc = nlp(sents[1])

for chunk in doc.noun_chunks:
    for token in chunk:
        token._.noun_chunk = chunk

for token in doc:
    print(token, "|", token.pos_, "|", token._.noun_chunk)

I | PRON | I
was | AUX | None
thrilled | ADJ | None
to | PART | None
be | AUX | None
back | ADV | None
in | ADP | None
the | DET | the Great city
Great | ADJ | the Great city
city | NOUN | the Great city
of | ADP | None
Charlotte | PROPN | Charlotte
, | PUNCT | None
North | PROPN | North Carolina
Carolina | PROPN | North Carolina
with | ADP | None
thousands | NOUN | thousands
of | ADP | None
hardworking | ADJ | hardworking American Patriots
American | ADJ | hardworking American Patriots
Patriots | PROPN | hardworking American Patriots
who | PRON | who
love | VERB | None
our | PRON | our Country
Country | PROPN | our Country
, | PUNCT | None
cherish | VERB | None
our | PRON | our values
values | NOUN | our values
, | PUNCT | None
respect | VERB | None
our | PRON | our laws
laws | NOUN | our laws
, | PUNCT | None
and | CCONJ | None
always | ADV | None
put | VERB | None
AMERICA | PROPN | AMERICA FIRST
FIRST | PROPN | AMERICA FIRST
! | PUNCT | None


Ou partir du mot et retraçant l'arbre de dépendance. C'est ce que vous ferez pour créer vos triplets.

La documentation est ici : https://spacy.io/usage/linguistic-features/#navigating

In [10]:
doc = nlp(sents[1])

for token in doc:
    if token.pos_ == "NOUN":
        print(token)
        print([t for t in token.subtree])
        print([t for t in token.children])
        print(token.head)
        print([t for t in token.lefts])
        print([t for t in token.rights])

city
[the, Great, city, of, Charlotte, ,, North, Carolina]
[the, Great, of]
in
[the, Great]
[of]
thousands
[thousands, of, hardworking, American, Patriots, who, love, our, Country]
[of]
with
[]
[of]
values
[our, values]
[our]
cherish
[our]
[]
laws
[our, laws]
[our]
respect
[our]
[]


In [11]:
doc = nlp(sents[1])

for token in doc:
    if token.pos_ == "VERB":
        print(token)
        print([t for t in token.subtree])
        print([t for t in token.children])
        print(token.head)
        print([t for t in token.lefts])
        print([t for t in token.rights])

love
[who, love, our, Country]
[who, Country]
Patriots
[who]
[Country]
cherish
[,, cherish, our, values, ,, respect, our, laws, ,, and, always, put, AMERICA, FIRST, !]
[,, values, ,, respect, !]
was
[,]
[values, ,, respect, !]
respect
[respect, our, laws, ,, and, always, put, AMERICA, FIRST]
[laws, ,, and, put]
cherish
[]
[laws, ,, and, put]
put
[always, put, AMERICA, FIRST]
[always, FIRST]
respect
[always]
[FIRST]


In [12]:
displacy.render(doc, style='dep', options={ "add_lemma": True})

In [13]:
for token in doc:
    print(token, "|", token.pos_, "|", token.dep_, "|", token.head)

I | PRON | nsubj | was
was | AUX | ROOT | was
thrilled | ADJ | acomp | was
to | PART | aux | be
be | AUX | xcomp | thrilled
back | ADV | advmod | be
in | ADP | prep | back
the | DET | det | city
Great | ADJ | amod | city
city | NOUN | pobj | in
of | ADP | prep | city
Charlotte | PROPN | pobj | of
, | PUNCT | punct | Charlotte
North | PROPN | compound | Carolina
Carolina | PROPN | appos | Charlotte
with | ADP | prep | be
thousands | NOUN | pobj | with
of | ADP | prep | thousands
hardworking | ADJ | amod | Patriots
American | ADJ | amod | Patriots
Patriots | PROPN | pobj | of
who | PRON | nsubj | love
love | VERB | relcl | Patriots
our | PRON | poss | Country
Country | PROPN | dobj | love
, | PUNCT | punct | cherish
cherish | VERB | dep | was
our | PRON | poss | values
values | NOUN | dobj | cherish
, | PUNCT | punct | cherish
respect | VERB | dep | cherish
our | PRON | poss | laws
laws | NOUN | dobj | respect
, | PUNCT | punct | respect
and | CCONJ | cc | respect
always | ADV | advmod |

In [14]:
for token in doc:
    print(token, token.ent_iob_, token.ent_type_)

I O 
was O 
thrilled O 
to O 
be O 
back O 
in O 
the O 
Great B GPE
city I GPE
of O 
Charlotte B GPE
, O 
North B GPE
Carolina I GPE
with O 
thousands B CARDINAL
of O 
hardworking O 
American B ORG
Patriots I ORG
who O 
love O 
our O 
Country B GPE
, O 
cherish O 
our O 
values O 
, O 
respect O 
our O 
laws O 
, O 
and O 
always O 
put O 
AMERICA O 
FIRST O 
! O 


Vous pouvez obtenir la liste des tags, entités et deps.

In [15]:
for label in nlp.get_pipe("parser").labels:
    print(label, " -- ", spacy.explain(label))

ROOT  --  root
acl  --  clausal modifier of noun (adjectival clause)
acomp  --  adjectival complement
advcl  --  adverbial clause modifier
advmod  --  adverbial modifier
agent  --  agent
amod  --  adjectival modifier
appos  --  appositional modifier
attr  --  attribute
aux  --  auxiliary
auxpass  --  auxiliary (passive)
case  --  case marking
cc  --  coordinating conjunction
ccomp  --  clausal complement
compound  --  compound
conj  --  conjunct
csubj  --  clausal subject
csubjpass  --  clausal subject (passive)
dative  --  dative
dep  --  unclassified dependent
det  --  determiner
dobj  --  direct object
expl  --  expletive
intj  --  interjection
mark  --  marker
meta  --  meta modifier
neg  --  negation modifier
nmod  --  modifier of nominal
npadvmod  --  noun phrase as adverbial modifier
nsubj  --  nominal subject
nsubjpass  --  nominal subject (passive)
nummod  --  numeric modifier
oprd  --  object predicate
parataxis  --  parataxis
pcomp  --  complement of preposition
pobj  --  ob

/home/laura/Documents/1_ertim/2_cours/2_semantique/s8_IE/ie_rel/lib/python3.10/site-packages/spacy/glossary.py:20: UserWarning: [W118] Term 'predet' not found in glossary. It may however be explained in documentation for the corpora used to train the language. Please check `nlp.meta["sources"]` for any relevant links.
  warnings.warn(Warnings.W118.format(term=term))


In [16]:
for label in nlp.get_pipe("tagger").labels:
    print(label, " -- ", spacy.explain(label))

$  --  symbol, currency
''  --  closing quotation mark
,  --  punctuation mark, comma
-LRB-  --  left round bracket
-RRB-  --  right round bracket
.  --  punctuation mark, sentence closer
:  --  punctuation mark, colon or ellipsis
ADD  --  email
AFX  --  affix
CC  --  conjunction, coordinating
CD  --  cardinal number
DT  --  determiner
EX  --  existential there
FW  --  foreign word
HYPH  --  punctuation mark, hyphen
IN  --  conjunction, subordinating or preposition
JJ  --  adjective (English), other noun-modifier (Chinese)
JJR  --  adjective, comparative
JJS  --  adjective, superlative
LS  --  list item marker
MD  --  verb, modal auxiliary
NFP  --  superfluous punctuation
NN  --  noun, singular or mass
NNP  --  noun, proper singular
NNPS  --  noun, proper plural
NNS  --  noun, plural
PDT  --  predeterminer
POS  --  possessive ending
PRP  --  pronoun, personal
PRP$  --  pronoun, possessive
RB  --  adverb
RBR  --  adverb, comparative
RBS  --  adverb, superlative
RP  --  adverb, particle


In [17]:
for label in nlp.get_pipe("ner").labels:
    print(label, " -- ", spacy.explain(label))

CARDINAL  --  Numerals that do not fall under another type
DATE  --  Absolute or relative dates or periods
EVENT  --  Named hurricanes, battles, wars, sports events, etc.
FAC  --  Buildings, airports, highways, bridges, etc.
GPE  --  Countries, cities, states
LANGUAGE  --  Any named language
LAW  --  Named documents made into laws.
LOC  --  Non-GPE locations, mountain ranges, bodies of water
MONEY  --  Monetary values, including unit
NORP  --  Nationalities or religious or political groups
ORDINAL  --  "first", "second", etc.
ORG  --  Companies, agencies, institutions, etc.
PERCENT  --  Percentage, including "%"
PERSON  --  People, including fictional
PRODUCT  --  Objects, vehicles, foods, etc. (not services)
QUANTITY  --  Measurements, as of weight or distance
TIME  --  Times smaller than a day
WORK_OF_ART  --  Titles of books, songs, etc.


In [18]:
spacy.explain("NN")

'noun, singular or mass'

In [19]:
spacy.explain("PERSON")

'People, including fictional'

# EXTRACT SVOs

In [21]:
sentences = sents[:20]

spos = []
sents_id = [] # index de la phrase dans la liste 'sents'

nlp = spacy.load("en_core_web_lg")
spacy_docs = nlp.pipe(sentences)

for i, doc in enumerate(spacy_docs):

    for token in doc:

        if token == "VERB":

            spo = {"s": [], "n": [], "p": [], "o": []}
        
            # d'abord le predicat
        
            # attraper la négation s'il y en a une (que vous pouvez garder sous une forme de lemme 'not')
        
            # réupérer l'objet

            # spo = (subj, neg, verb, obj)
            spos.append(spo)